# 07 - Proposed model, weak labels as an auxiliary stream

Notebook 06 trained all six rungs. This one retrains **only the rungs that use
weak labels** - `L3_weak`, `L4_contrastive`, `L5_full` - because those are the
only ones affected by the defect described below. `L0_flat`, `L1_bottleneck` and
`L2_graph` from your existing run stay valid and are reused as-is.

**9 runs instead of 18, about half the compute.**

## The defect

`load_frames` concatenated the weak rows into the training frame. For L3-L5 that
meant:

| | L0-L2 | L3-L5 (old) |
|---|---|---|
| train rows | 12,911 | 18,911 |
| optimiser steps / epoch | 404 | 591 |
| cosine LR schedule | over 4,040 steps | over 5,910 steps |
| rows per batch with a risk label | 0.84 | 0.58 |
| `pos_weight` computed over | gold only | gold + weak |

So L3-L5 ran a **different LR schedule** and a **thinner gold batch** than
L0-L2. The damage in your run landed exactly where that predicts - on the
sparsest tasks:

| task | train rows | L2 -> L3 | delta |
|---|---|---|---|
| symptoms | 6,227 | 0.896 -> 0.891 | -0.005 |
| stress | ~3,526 | 0.907 -> 0.905 | -0.002 |
| severity (depression) | 2,456 | 0.480 -> 0.443 | -0.037 |
| **risk** | **340** | **0.438 -> 0.249** | **-0.189** |

That is label dilution, not a failure of novelty N3.

## The change

Weak rows become a **per-step auxiliary symptom loss** drawn from a separate
loader. Steps per epoch, the LR schedule, `pos_weight` and the gold batch
composition all go back to being identical to L0-L2, so the comparison against
your existing L0-L2 numbers is valid.

Two side benefits:

- The cost of the stream is set by `weak_batch_size`, **not** by the size of the
  weak pool, so all ~50,000 real Qwen labels are used for the same price as the
  6,000 you used before.
- `pos_weight` is now computed on gold labels only, which is what L0-L2 used.

## What this notebook does not do

It does not change epochs, learning rates, the schedule, batch size, sequence
length, class weights, thresholds, the selection metric or the encoder. Those are
read back out of your existing `L2_graph` config and reused verbatim - section 3
asserts it. Changing any of them would mean the new L3-L5 numbers could not be
compared to the old L0-L2 numbers, which is the whole point of training only
three rungs.

**No promise is attached to the outcome.** The hypothesis is written to Drive
before training starts (section 4) and reported either way.

## 1. Mount Drive and configure

In [ ]:
import os, sys, json, math, warnings
import pandas as pd
from google.colab import drive
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/hari/SymptoGraph-LLM'
PROCESSED  = f'{DRIVE_BASE}/processed'
GRAPH      = f'{DRIVE_BASE}/graph'
WEAK       = f'{DRIVE_BASE}/weak_labels/weak_labels.parquet'
BASELINES_DIR = f'{DRIVE_BASE}/baselines'

# Existing notebook-06 output. L0-L2 are read from here and NOT retrained.
RESULTS_V5 = f'{DRIVE_BASE}/proposed'
# New output for the stream-trained rungs. Separate folder so the valid L0-L2
# results cannot be overwritten or mixed up.
RESULTS    = f'{DRIVE_BASE}/proposed_stream'

REUSED = ['L0_flat', 'L1_bottleneck', 'L2_graph']
RUNGS  = ['L3_weak', 'L4_contrastive', 'L5_full']

os.makedirs(RESULTS, exist_ok=True)
print('reusing  :', REUSED, '->', RESULTS_V5)
print('training :', RUNGS, '->', RESULTS)

for p in (PROCESSED, GRAPH, RESULTS_V5):
    assert os.path.isdir(p), f'missing {p}'
assert os.path.exists(WEAK), f'missing {WEAK} - run notebook 04'

for r in REUSED:
    f = f'{RESULTS_V5}/{r}/aggregate/aggregate_metrics.json'
    assert os.path.isfile(f), (f'{r} has no aggregate results in {RESULTS_V5}. '
                               'This notebook reuses them, so notebook 06 must '
                               'have finished those three rungs.')
print('all three reused rungs have aggregate results')

w = pd.read_parquet(WEAK)
kinds = sorted({str(k) for k in w.get('labeller', pd.Series(dtype=str)).dropna().unique()})
print(f'\nweak labels: {len(w):,} rows, labeller {kinds}')
assert not any('mock' in k.lower() for k in kinds), 'mock weak labels - re-run notebook 04'


## 2. Get the code

In [ ]:
REPO = 'https://github.com/sid-2k6/SymptoGraph-LLM.git'
CHECKOUT = '/content/SymptoGraph-LLM'

if os.path.isdir(f'{CHECKOUT}/.git'):
    !git -C $CHECKOUT fetch -q origin && git -C $CHECKOUT reset -q --hard origin/main
else:
    !git clone -q --branch main $REPO $CHECKOUT

sys.path.insert(0, f'{CHECKOUT}/src')
from symptograph import proposed as P
assert hasattr(P, 'cycle_loader'), 'stale checkout - run: !rm -rf /content/SymptoGraph-LLM'
!git -C $CHECKOUT log --oneline -1


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> L4 GPU + High-RAM.'
gpu = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'{gpu}  {total_gb:.0f} GB')


## 3. Rebuild the exact protocol from the existing run

The config is **loaded from `L2_graph/seed_0/config.json`**, not retyped. Only
the weak-stream fields are overridden. This makes protocol parity true by
construction rather than by hand-checking, and the assertion below proves it.

In [ ]:
from dataclasses import fields as dc_fields
from symptograph.proposed import ProposedConfig, LADDER, LADDER_ORDER, FULL_MODEL
from symptograph.jsonio import read_json

old = read_json(f'{RESULTS_V5}/L2_graph/seed_0/config.json')
allowed = {f.name for f in dc_fields(ProposedConfig)}
kw = {k: v for k, v in old.items() if k in allowed}
kw['seeds'] = tuple(kw['seeds'])

# The only intentional differences. Everything else is inherited verbatim.
OVERRIDES = dict(
    weak_label_path=WEAK,
    weak_as_stream=True,     # the fix
    weak_max_rows=None,      # all ~50k labels; free under the stream design
    weak_batch_size=16,      # budget lever: 16 keeps 9 runs near 3.5 h on an L4
    weak_every=1,
    weak_stream_weight=1.0,
)
cfg = ProposedConfig(**{**kw, **OVERRIDES})

# Fields allowed to differ: the component switches (those ARE the rungs), the
# weak-stream mechanics, and pure I/O settings that cannot affect the maths.
MAY_DIFFER = {'use_bottleneck', 'use_graph', 'use_weak', 'use_contrastive',
              'use_uncertainty', 'weak_as_stream', 'weak_batch_size',
              'weak_every', 'weak_stream_weight', 'weak_max_rows',
              'weak_label_path', 'allow_mock_weak_labels',
              'checkpoint_every', 'save_optimizer_state'}

drift = []
for f in sorted(allowed - MAY_DIFFER):
    a, b = old.get(f), getattr(cfg, f)
    if isinstance(b, tuple):
        a = tuple(a) if isinstance(a, list) else a
    if a != b:
        drift.append({'field': f, 'L2_graph_run': a, 'this_run': b})

if drift:
    display(pd.DataFrame(drift))
    raise AssertionError('Protocol drift against the existing L0-L2 run. Those '
                         'rungs would no longer be comparable - fix before training.')
print('protocol parity with the existing L0-L2 run: OK')
print(f'  epochs {cfg.epochs} | seeds {cfg.seeds} | batch {cfg.batch_size} | '
      f'max_length {cfg.max_length}')
print(f'  lr {cfg.lr}/{cfg.head_lr} | {cfg.scheduler} | patience '
      f'{cfg.early_stopping_patience} | selection {cfg.selection_metric}')
print(f'  pos_weight_cap {cfg.pos_weight_cap} | ordinal weights '
      f'{cfg.ordinal_class_weights} | tuned thresholds {cfg.tune_thresholds}')
print(f'  encoder {cfg.encoder_name}')
print()
print('intentional differences:')
for k, v in OVERRIDES.items():
    print(f'  {k:20s} {old.get(k)!r:>12} -> {v!r}')


In [ ]:
# The encoder is gated; fail here rather than on the first rung.
from symptograph.proposed import check_encoder_access

status = check_encoder_access(cfg.encoder_name)
if not status['ok']:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        from getpass import getpass
        token = getpass('Hugging Face read token: ').strip() or None
    if token:
        from huggingface_hub import login
        login(token=token, add_to_git_credential=False)
        status = check_encoder_access(cfg.encoder_name)
assert status['ok'], f"no access to {cfg.encoder_name}: {status['reason']}"
print('encoder access OK:', cfg.encoder_name)


## 4. Pre-registered hypothesis

Written to Drive **before** training. The point is that the result counts
whichever way it goes: if L5 does not clear L2 after the confound is removed,
that is a finding about the components, not something to re-tune until it looks
better.

In [ ]:
from symptograph.jsonio import write_json

baseline_numbers = {}
for r in REUSED:
    agg = read_json(f'{RESULTS_V5}/{r}/aggregate/aggregate_metrics.json')['aggregate']
    baseline_numbers[r] = {k: round(v['mean'], 4) for k, v in agg.items()
                           if k in ('composite', 'selection_score',
                                    'severity_risk_qwk', 'severity_depression_qwk',
                                    'symptom_macro_auroc[all]', 'stress_auroc')}

HYPOTHESIS = {
    'defect': 'weak rows were concatenated into the training frame, inflating '
              'steps/epoch by 46%, changing the LR schedule, recomputing '
              'pos_weight over weak rows, and thinning risk labels per batch '
              'from 0.84 to 0.58',
    'change': 'weak rows become a per-step auxiliary symptom loss; steps/epoch, '
              'schedule, pos_weight and gold batch composition return to L0-L2 values',
    'predictions': [
        'L3_weak recovers most of the 0.189 risk-QWK loss it showed against L2',
        'L3_weak composite returns to at least L2 level (0.571), vs 0.494 before',
        'symptom metrics improve modestly from using ~50k weak labels instead of 6k',
        'L4/L5 gains over L3 remain small and may stay inside seed noise, since '
        'contrastive and uncertainty were near noise in the previous run',
    ],
    'not_predicted': [
        'any metric reaching 0.85 that is not already there (stress AUROC and '
        'symptom AUROC already exceed it; severity and risk QWK will not)',
        'a guaranteed margin of L5 over every other rung',
    ],
    'falsification': 'if L3_weak still underperforms L2 after this change, the '
                     'dilution hypothesis is wrong and N3 does not help on the '
                     'currently measurable symptoms',
    'measurement_gap': 'symptom_macro_auroc[all] equals '
                       'symptom_macro_auroc[gold_supervised] because the 11 '
                       'weak-only symptoms have no gold labels in any split. N3 '
                       'therefore cannot be credited directly by the current '
                       'metric set - only through transfer to the 11 gold '
                       'symptoms. The 800-post in-house set from notebook 04 '
                       'section 8 is what would close this.',
    'reused_rungs': baseline_numbers,
    'overrides': {k: str(v) for k, v in OVERRIDES.items()},
}
write_json(f'{RESULTS}/hypothesis.json', HYPOTHESIS)
print(json.dumps(HYPOTHESIS, indent=2)[:2000])


## 5. Budget

9 runs. The stream adds one extra encoder pass per step, sized by
`weak_batch_size`.

| setting | cost vs L2 per epoch | 9 runs on an L4 |
|---|---|---|
| `weak_batch_size=16, weak_every=1` (default) | ~1.5x | ~3.5 h |
| `weak_batch_size=32, weak_every=1` | ~2.0x | ~4.5 h |
| `weak_batch_size=16, weak_every=2` | ~1.25x | ~3 h |

Set it **before** starting, so all three rungs match. Everything resumes and
writes straight to Drive, so a disconnect costs at most the epoch in progress;
`RUNGS` in cell 1 splits the work across sessions.

## 6. Train

In [ ]:
from symptograph.preprocess import Ontology
from symptograph.metrics import compare_runs, headline_table, set_publication_style

onto = Ontology.load(f'{CHECKOUT}/configs/symptom_ontology.json')
set_publication_style(font_size=20)

results = P.run_ladder(cfg, PROCESSED, GRAPH, RESULTS,
                       rungs=RUNGS, onto=onto, resume=True, verbose=True)
print('\ntrained:', list(results))


## 7. Integrity checks

Three mechanical checks that the thing I claimed to fix is actually fixed.
Run these before looking at any metric.

In [ ]:
from symptograph.jsonio import repair_json_tree
_ = repair_json_tree(RESULTS)

# (a) the weak stream actually ran, on every step
print('(a) weak stream')
for r in RUNGS:
    h = pd.read_csv(f'{RESULTS}/{r}/seed_0/history.csv')
    steps = h['train_weak_steps'].iloc[0]
    print(f'    {r:16s} weak batches/epoch {steps}  '
          f'mean weak loss {h["train_weak_loss"].mean():.4f}')
    assert steps and steps > 0, f'{r} ran no weak batches - the stream is not wired up'

# (b) the LR schedule is now identical to L2's. Same steps/epoch and same total
#     steps means the encoder LR at the end of every epoch must match exactly.
print('\n(b) LR schedule parity with L2_graph')
l2 = pd.read_csv(f'{RESULTS_V5}/L2_graph/seed_0/history.csv')[['epoch', 'lr_encoder']]
new = pd.read_csv(f'{RESULTS}/L3_weak/seed_0/history.csv')[['epoch', 'lr_encoder']]
cmp_lr = l2.merge(new, on='epoch', suffixes=('_L2', '_L3_new'))
cmp_lr['abs_diff'] = (cmp_lr.lr_encoder_L2 - cmp_lr.lr_encoder_L3_new).abs()
display(cmp_lr.round(10))
worst = cmp_lr.abs_diff.max()
print(f'    max difference {worst:.3e}')
assert worst < 1e-9, ('LR schedules still differ, so L3-L5 are not comparable '
                      'to L0-L2. Check steps per epoch.')

# (c) pos_weight was computed on gold rows only
print('\n(c) gold-only pos_weight')
c3 = read_json(f'{RESULTS}/L3_weak/seed_0/config.json')
print(f"    n_weak_rows recorded {c3['n_weak_rows']:,} (stream, not concatenated)")
print(f"    weak_as_stream {c3['weak_as_stream']}")
assert c3['weak_as_stream'] is True
print('\nall integrity checks passed')


## 7b. Did the fix change anything, and is the model overfitting?

Two separate questions, both answered from `history.csv` with no GPU.

The first compares old and new `L3_weak` **at matched epochs**, which separates
the effect of the dilution fix from the fact that the best epoch may have moved.
The second checks whether the epoch that `selection_score` picked is one you
would actually want to ship.

In [ ]:
# --- old vs new L3_weak, matched epochs -------------------------------------
COLS = ['epoch', 'val_loss', 'val_selection_score', 'val_composite',
        'val_group_all_macro_f1', 'val_group_all_macro_auroc',
        'val_severity_depression_qwk', 'val_severity_risk_qwk']

frames = []
for seed in cfg.seeds:
    o = pd.read_csv(f'{RESULTS_V5}/L3_weak/seed_{seed}/history.csv')[COLS]
    n = pd.read_csv(f'{RESULTS}/L3_weak/seed_{seed}/history.csv')[COLS]
    frames.append(o.merge(n, on='epoch', suffixes=('_old', '_new')))
matched = pd.concat(frames).groupby('epoch').mean().round(4)

for m in ('val_composite', 'val_severity_risk_qwk', 'val_severity_depression_qwk',
          'val_group_all_macro_auroc'):
    sub = matched[[f'{m}_old', f'{m}_new']].copy()
    sub['delta'] = (sub[f'{m}_new'] - sub[f'{m}_old']).round(4)
    print(f'\n{m}  (mean over seeds, same epoch number on both sides)')
    print(sub.to_string())


In [ ]:
# --- is the selected epoch overfitted? --------------------------------------
rows = []
for rung in LADDER_ORDER:
    for src, tag in ((RESULTS, 'new'), (RESULTS_V5, 'v5')):
        for seed in cfg.seeds:
            f = f'{src}/{rung}/seed_{seed}/history.csv'
            if not os.path.isfile(f):
                continue
            h = pd.read_csv(f)
            sel = h.loc[h.val_selection_score.idxmax()]
            lo = h.loc[h.val_loss.idxmin()]
            co = h.loc[h.val_composite.idxmax()]
            rows.append({
                'rung': rung, 'run': tag, 'seed': seed, 'epochs_run': len(h),
                'ep_by_selection': int(sel.epoch), 'ep_by_val_loss': int(lo.epoch),
                'ep_by_composite': int(co.epoch),
                'val_loss_at_selected': round(sel.val_loss, 3),
                'val_loss_min': round(lo.val_loss, 3),
                'loss_inflation': round(sel.val_loss / lo.val_loss, 2),
                'riskQWK_at_selected': round(sel.val_severity_risk_qwk, 3),
                'riskQWK_best': round(h.val_severity_risk_qwk.max(), 3),
                'train_riskQWK_final': round(h.train_severity_risk_qwk.iloc[-1], 3),
            })
diag = pd.DataFrame(rows)
display(diag.set_index(['rung', 'run', 'seed']))
diag.to_csv(f'{RESULTS}/selection_diagnostics.csv', index=False)

bad = diag[diag.loss_inflation > 1.3]
if len(bad):
    print(f'{len(bad)} of {len(diag)} runs selected a checkpoint whose validation '
          f'loss is >1.3x the minimum reached.')
    print('selection_score = mean(symptom macro F1, symptom macro AUROC, severity '
          'QWK).\nTwo of its three terms are symptom metrics, and those keep '
          'climbing while\nthe ordinal heads memorise their small training sets - '
          'so the score rises on\nepochs you would not ship. train_riskQWK_final '
          'near 1.000 is that memorisation.')
else:
    print('No run selected a badly overfitted checkpoint.')


## 8. The combined ladder

L0-L2 from the existing run, L3-L5 from this one. Valid because section 3 proved
the protocols match and section 7 proved the schedule now matches.

In [ ]:
runs = {r: f'{RESULTS_V5}/{r}' for r in REUSED}
runs.update({r: f'{RESULTS}/{r}' for r in RUNGS
             if os.path.isfile(f'{RESULTS}/{r}/aggregate/aggregate_metrics.json')})

combined, tables = compare_runs(runs, f'{RESULTS}/comparison_ladder',
                                title='SymptoGraph-LLM ablation ladder',
                                font_size=20)
display(combined.pivot(index='metric', columns='model', values='formatted'))


In [ ]:
# per-rung delta on the selection metric and on composite, with the
# seed standard deviation alongside, so "gain" can be read against noise
rows, prev = [], {}
order = [r for r in LADDER_ORDER if r in runs]
for r in order:
    agg = read_json(f'{runs[r]}/aggregate/aggregate_metrics.json')['aggregate']
    rec = {'rung': r, 'protocol': 'v5 (reused)' if r in REUSED else 'stream (new)'}
    for m in ('selection_score', 'composite'):
        if m in agg:
            rec[m] = round(agg[m]['mean'], 4)
            rec[f'{m}_std'] = round(agg[m]['std'], 4)
            rec[f'{m}_delta'] = (None if m not in prev
                                 else round(agg[m]['mean'] - prev[m], 4))
            prev[m] = agg[m]['mean']
    rows.append(rec)
ladder = pd.DataFrame(rows).set_index('rung')
display(ladder)
ladder.to_csv(f'{RESULTS}/comparison_ladder/ladder_deltas.csv')

print('Read a delta as real only if it exceeds the std of both rungs it spans.')


## 9. Per-task results

`composite` averages suicide-risk QWK, measured on **74 test rows**, in with
everything else, so one very small task moves the headline by more than it
should. Report per task.

In [ ]:
TASKS = ['symptom_macro_auroc[all]', 'symptom_macro_f1[all]',
         'stress_auroc', 'stress_f1',
         'severity_depression_qwk', 'severity_depression_mae',
         'severity_risk_qwk', 'severity_risk_mae']
EVAL_ROWS = {'symptom': '1,956 test posts', 'stress': '~700 test posts',
             'severity_depression': '541 test posts', 'severity_risk': '74 test posts'}

rows = []
for r in order:
    agg = read_json(f'{runs[r]}/aggregate/aggregate_metrics.json')['aggregate']
    rec = {'rung': r}
    for m in TASKS:
        if m in agg:
            rec[m] = f"{agg[m]['mean']:.3f} +/- {agg[m]['std']:.3f}"
    rows.append(rec)
display(pd.DataFrame(rows).set_index('rung').T)

print('test-set sizes behind each task:')
for k, v in EVAL_ROWS.items():
    print(f'  {k:22s} {v}')
print('\nlower is better: severity_depression_mae, severity_risk_mae')


## 10. Against the baselines

In [ ]:
from symptograph.baselines import BASELINES

bl = {b: f'{BASELINES_DIR}/{b}' for b in BASELINES
      if os.path.isfile(f'{BASELINES_DIR}/{b}/aggregate/aggregate_metrics.json')}
if not bl:
    print(f'No baseline results in {BASELINES_DIR} - run notebook 05.')
else:
    vs = dict(bl)
    vs['SymptoGraph-LLM'] = f'{RESULTS}/{FULL_MODEL}'
    cb, tb = compare_runs(vs, f'{RESULTS}/comparison_vs_baselines',
                          title='SymptoGraph-LLM vs baselines', font_size=20)
    display(cb.pivot(index='metric', columns='model', values='formatted'))

    print('\nNote for the paper: L3-L5 see ~50k LLM weak labels that the')
    print('baselines never saw. Part of any margin is extra supervision, not')
    print('architecture. Re-run notebook 05 with weak labels enabled for a')
    print('matched-data comparison before claiming the architecture alone wins.')


In [ ]:
# held-out test, from each seed's best-validation checkpoint
rows = []
for label, path in {**bl, 'SymptoGraph-LLM': f'{RESULTS}/{FULL_MODEL}'}.items():
    f = f'{path}/aggregate/aggregate_metrics.json'
    if os.path.isfile(f):
        by = read_json(f).get('aggregate_by_split', {})
        if 'test' in by:
            rows.append(headline_table(by['test'], label=label))
if rows:
    tc = pd.concat(rows, ignore_index=True)
    display(tc.pivot(index='metric', columns='model', values='formatted'))
    tc.to_csv(f'{RESULTS}/comparison_vs_baselines/test_comparison.csv', index=False)


## 11. Reading the result

Compare what happened against `hypothesis.json`, written before training.

**If L3 recovered and the ladder rises** - the dilution diagnosis was right.
Report the defect and the fix in the paper's experimental section; it explains
why N3 appeared not to work and is a legitimate methodological contribution.

**If L3 recovered but L4/L5 add nothing beyond it** - report that honestly. "The
graph and calibrated weak supervision carry the gain; contrastive and
uncertainty did not help on these benchmarks" is a credible, publishable result.
A ladder where every one of five components conveniently helps is the version
reviewers distrust.

**If L3 still underperforms L2** - the hypothesis is falsified and the weak
labels genuinely do not help the measurable symptoms. The reason is in
`hypothesis.json` under `measurement_gap`: the 11 weak-only symptoms have no
gold labels anywhere, so N3 can only be credited through transfer. Closing that
needs the 800-post annotation set from notebook 04 section 8, not more training.

**On the 0.85 target.** `stress_auroc` (~0.91) and `symptom_macro_auroc`
(~0.89) are already above it and are the right headline numbers for a detection
paper. `severity_risk_qwk` will not approach 0.85 from 340 training rows, and a
paper reporting that it did would be read as a leakage bug. The MAE metrics are
lower-is-better, so "above 0.85" would make them worse.

**Still outstanding for the paper:** N4 (rationale generation and its
faithfulness evaluation), the concept-intervention experiment, external
evaluation on the held-out IMHI subsets, and the in-house anxiety gold set.